## 4. Entwurf des mathematischen Modells: Gefahrgut-Routenplanung & Zuweisung

**Modelltyp:** Gemischt-ganzzahliges lineares Programm (MILP)

Dieses Modell löst ein kombiniertes Zuweisungs- und Netzwerkflussproblem. Da wir diskrete Ja/Nein-Entscheidungen treffen müssen (z. B. ein Fahrzeug zuweisen, einen bestimmten Weg nehmen), verwenden wir binäre Entscheidungsvariablen[cite: 2].

### 4.1 Sets (Mengen)
* **$V$**: Menge der verfügbaren Fahrzeuge (z. B. MAN_eTGX, Volvo_FH_Electric).
* **$L$**: Menge der durchzuführenden Gefahrgutlieferungen.
* **$N$**: Menge aller Knoten (Kreuzungen, Start-/Zielpunkte) im Straßennetz.
* **$E$**: Menge aller gerichteten Kanten (Straßenabschnitte) im Netzwerk. Eine Kante $e$ verläuft von Knoten $i$ nach Knoten $j$.
* **$K$**: Menge der Gefahrgutklassen.

### 4.2 Parameters (Parameter)
* **$Cap_v$**: Maximale Nutzlastkapazität des Fahrzeugs $v$ in Tonnen.
* **$FC_v$**: Fixkosten für den Einsatz des Fahrzeugs $v$ für einen Tag.
* **$VC_{v, e}$**: Variable Transportkosten für Fahrzeug $v$ auf Kante $e$ (basierend auf Distanz, Energieverbrauch und lokalen Energiepreisen).
* **$Dem_l$**: Gewicht/Bedarf der Lieferung $l$ in Tonnen.
* **$Class_l$**: Die spezifische Gefahrgutklasse $k \in K$ der Lieferung $l$.
* **$O_l, D_l$**: Startknoten (Origin) und Zielknoten (Destination) der Lieferung $l$.
* **$Risk_{e, k}$**: Der berechnete Risikoscore für Kante $e$ und Gefahrgutklasse $k$ (aggregiert Bevölkerungsdichte, Naturschutzgebiete, Unfallwahrscheinlichkeit).
* **$Allow_{e, k} \in \{0, 1\}$**: 1, wenn Kante $e$ für die Gefahrgutklasse $k$ rechtlich freigegeben ist, sonst 0.

### 4.3 Decision Variables (Entscheidungsvariablen)
Binäre Variablen dienen als Wahrheitswerte für unsere logischen Aussagen[cite: 2].

* **$x_{l, v, e} \in \{0, 1\}$**: 1, wenn Lieferung $l$ von Fahrzeug $v$ über die Kante $e$ transportiert wird, sonst 0. 
  * *Bedeutung in der Praxis: "Fährt genau dieser LKW mit genau dieser Lieferung über genau diesen Straßenabschnitt?"*
* **$y_{l, v} \in \{0, 1\}$**: 1, wenn Lieferung $l$ dem Fahrzeug $v$ zugewiesen wird, sonst 0.
  * *Bedeutung in der Praxis: "Übernimmt LKW v die Lieferung l?"*
* **$z_v \in \{0, 1\}$**: 1, wenn Fahrzeug $v$ überhaupt eingesetzt wird, sonst 0.
  * *Bedeutung in der Praxis: "Verlässt dieser LKW heute das Depot?"*

### 4.4 Objective Function (Zielfunktion)
Wir wollen sowohl das Risiko als auch die Kosten durch einen gewichteten Summenansatz (Skalarisierung) minimieren.

$$ \min \quad w_1 \sum_{l \in L} \sum_{v \in V} \sum_{e \in E} Risk_{e, Class_l} \cdot x_{l, v, e} \quad + \quad w_2 \left( \sum_{v \in V} FC_v \cdot z_v + \sum_{l \in L} \sum_{v \in V} \sum_{e \in E} VC_{v, e} \cdot x_{l, v, e} \right) $$

### 4.5 Main Constraints (Nebenbedingungen)

**1. Transportpflicht (Assignment Constraint):**
Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
$$ \sum_{v \in V} y_{l, v} = 1 \quad \forall l \in L $$

**2. Kapazitäts- & Aktivierungsbedingung (Capacity & Activation Constraint):**
Ein Fahrzeug kann nur Lieferungen bis zu seiner Kapazitätsgrenze aufnehmen. Außerdem muss das Fahrzeug aktiviert werden ($z_v = 1$), wenn ihm eine Lieferung zugewiesen wird. Dies ist eine klassische Aktivierungsbedingung ("Big-M")[cite: 2].
$$ \sum_{l \in L} Dem_l \cdot y_{l, v} \le Cap_v \cdot z_v \quad \forall v \in V $$

**3. Straßennetz- & Gefahrgutrestriktionen (Street Network & Hazard Restrictions):**
Ein Fahrzeug darf eine Kante nur befahren, wenn sie für das Gefahrgut der aktuellen Lieferung rechtlich freigegeben ist.
$$ x_{l, v, e} \le Allow_{e, Class_l} \quad \forall l \in L, v \in V, e \in E $$

**4. Routenlogik / Flusserhaltung (Flow Conservation):**
Stellt sicher, dass ein LKW, der in eine Kreuzung ($n$) einfährt, diese auch wieder verlassen muss, es sei denn, es handelt sich um den Start ($O_l$) oder das Ziel ($D_l$) der Lieferung.
$$ \sum_{e \in \text{Eingehend}(n)} x_{l, v, e} - \sum_{e \in \text{Ausgehend}(n)} x_{l, v, e} = 
\begin{cases} 
-y_{l, v} & \text{wenn } n = O_l \\ 
y_{l, v} & \text{wenn } n = D_l \\ 
0 & \text{sonst} 
\end{cases} \quad \forall l \in L, v \in V, n \in N $$

**5. Koppelung von Kante und Zuweisung (Edge-Assignment Linkage):**
Eine Route für Lieferung $l$ mit Fahrzeug $v$ kann nur gebildet werden, wenn die Lieferung diesem Fahrzeug auch tatsächlich zugewiesen wurde.
$$ x_{l, v, e} \le y_{l, v} \quad \forall l \in L, v \in V, e \in E $$

In [2]:
import pulp as pl

# 1. Initialisiere das Modell (Minimierungsproblem)
model = pl.LpProblem("Hazardous_Material_Routing_Optimized", pl.LpMinimize)

# --- DUMMY DATEN (Mit euren echten Daten aus dem data-Branch ersetzen) ---
L = ['L1', 'L2']                     # Lieferungen
V = ['MAN_eTGX', 'Volvo_FH_Electric']# Fahrzeuge
N = ['A', 'B', 'C', 'D']             # Knoten
E = [('A', 'B'), ('B', 'C'), ('C', 'D'), ('A', 'C')] # Kanten
K = ['Klasse_3', 'Klasse_8']         # Gefahrgutklassen

O = {'L1': 'A', 'L2': 'B'}           # Startknoten
D = {'L1': 'D', 'L2': 'C'}           # Zielknoten
Dem = {'L1': 10, 'L2': 15}           # Gewicht
Class = {'L1': 'Klasse_3', 'L2': 'Klasse_8'} 

Cap = {'MAN_eTGX': 18, 'Volvo_FH_Electric': 20}
FC = {'MAN_eTGX': 180, 'Volvo_FH_Electric': 200}
VC = {v: {e: 1.5 for e in E} for v in V} # Variable Kosten
Risk = {e: {k: 50 for k in K} for e in E} # Risikoscore
Allow = {e: {k: 1 for k in K} for e in E} # Erlaubnis (1=Ja, 0=Nein)

# --- 2. ENTSCHEIDUNGSVARIABLEN ---

# FIX 1: Entkopplung von Routing und Zuweisung (verhindert Variablenexplosion)
# f[l, e]: Fließt Lieferung l über Kante e? (Fahrzeug-agnostisch)
f = pl.LpVariable.dicts("Fluss", 
                        [(l, e) for l in L for e in E], 
                        cat='Binary')

# y[l, v]: Wird Lieferung l dem Fahrzeug v zugewiesen?
y = pl.LpVariable.dicts("Zuweisung", 
                        [(l, v) for l in L for v in V], 
                        cat='Binary')

# z[v]: Wird Fahrzeug v eingesetzt?
z = pl.LpVariable.dicts("Aktivierung", 
                        [v for v in V], 
                        cat='Binary')

# FIX 2: MTZ-Hilfsvariablen zur Subtour-Eliminierung
# u[l, n]: Besuchsreihenfolge für Lieferung l an Knoten n
u = pl.LpVariable.dicts("MTZ", 
                        [(l, n) for l in L for n in N], 
                        lowBound=0, upBound=len(N), cat='Continuous')


# --- 3. ZIELFUNKTION ---

w1, w2 = 0.5, 0.5

# FIX 5: Normalisierung der Zielfunktion
# Schätze Max-Werte, um Risiko und Kosten auf eine ähnliche Skala (0 bis 1) zu bringen
Risk_max = max(Risk[e][k] for e in E for k in K) * len(E) # Worst-Case Risiko
Cost_max = sum(FC[v] for v in V) + sum(VC[v][e] for v in V for e in E) # Worst-Case Kosten

# Vermeide Division durch Null
Risk_max = Risk_max if Risk_max > 0 else 1
Cost_max = Cost_max if Cost_max > 0 else 1

model += (
    w1 * pl.lpSum(Risk[e][Class[l]] * f[(l, e)] for l in L for e in E) / Risk_max +
    w2 * (pl.lpSum(FC[v] * z[v] for v in V) + 
          pl.lpSum(VC[v][e] * y[(l,v)] for l in L for v in V for e in E)) / Cost_max
), "Gewichtete_Zielfunktion"


# --- 4. NEBENBEDINGUNGEN ---

# Constraint 1: Transportpflicht
for l in L:
    model += pl.lpSum(y[(l, v)] for v in V) == 1, f"Transportpflicht_{l}"

# Constraint 2 & FIX 3: Kapazität und explizite Aktivierung
for v in V:
    # Kapazität
    model += pl.lpSum(Dem[l] * y[(l, v)] for l in L) <= Cap[v] * z[v], f"Kapazitaet_{v}"
    
    # Logische Aktivierung: Wenn l an v zugewiesen ist, muss z[v] = 1 sein
    for l in L:
        model += z[v] >= y[(l, v)], f"Aktivierung_aus_Zuweisung_{l}_{v}"

# FIX 4: Gefahrgutrestriktionen aktivieren
for l in L:
    for e in E:
        model += f[(l, e)] <= Allow[e][Class[l]], f"Erlaubnis_{l}_{e[0]}_{e[1]}"

# FIX 6: Flusserhaltung (ausprogrammiert auf Basis der entkoppelten Variable f)
for l in L:
    for n in N:
        inflow  = pl.lpSum(f[(l, e)] for e in E if e[1] == n)
        outflow = pl.lpSum(f[(l, e)] for e in E if e[0] == n)
        
        if n == O[l]:
            model += outflow - inflow == 1, f"Flow_Start_{l}_{n}"
        elif n == D[l]:
            model += inflow - outflow == 1, f"Flow_Ziel_{l}_{n}"
        else:
            model += inflow == outflow, f"Flow_Transit_{l}_{n}"

# FIX 2: MTZ Subtour-Eliminierung
for l in L:
    for e in E:
        i, j = e[0], e[1]
        if j != O[l]:  # MTZ gilt nicht für den Startknoten
            model += u[(l, i)] - u[(l, j)] + len(N) * f[(l, e)] <= len(N) - 1, f"MTZ_{l}_{i}_{j}"


# 5. Modell lösen (Beispiel)
# model.solve()
# print(f"Status: {pl.LpStatus[model.status]}")